<a href="https://colab.research.google.com/github/suzukanitta-sketch/app_retention/blob/main/retention_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install opendatasets

In [ ]:
import opendatasets as od
od.download("https://www.kaggle.com/datasets/jayaantanaath/synthetic-user-event-log-object-datetime-format")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: suzuka-n
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/jayaantanaath/synthetic-user-event-log-object-datetime-format


100%|██████████| 1.59k/1.59k [00:00<00:00, 4.27MB/s]

In [ ]:
import pandas as pd
data = pd.read_csv("/content/synthetic-user-event-log-object-datetime-format/datetime.csv")
data.shape
data.head()

,event_id,timestamp,event_type,user_id
0,E001,2024-11-03 23:10:48,Purchase,U6516
1,E002,2023-06-15 23:10:26,Error,U7154
2,E003,2024-04-13 10:00:02,Logout,U5019
3,E004,2024-07-04 06:33:55,Purchase,U5808
4,E005,2024-02-20 01:22:45,Purchase,U1036


**Problem Definition**
Goal: Evaluate impact of new feature on 30-day retention

Decision: Ship / No-Ship

Risks: harming engagement or specific user groups


**Metric Design**
Primary metric
30-day retention

Guardrails
average sessions per user

In [ ]:
print("Event Type Distribution:")
print(data['event_type'].value_counts())


Event Type Distribution:
event_type
Purchase    36
Login       23
Logout      21
Error       20
Name: count, dtype: int64


In [ ]:
# retention: user had any activity between day 1–30 after first activity

#check if user had multiple activities
data.groupby('user_id').size()

#since there is only one event per user, I am going to simulate the repeated behavior
import numpy as np
import pandas as pd
np.random.seed(42)

# Define possible event types
event_types = ['Login', 'Purchase', 'View_Product', 'Logout']

simulated = []

for user in data['user_id'].unique():
  num_events = np.random.randint(1, 10)
  base_time = pd.Timestamp('2024-01-01')

  for i in range(num_events):
    simulated.append({
        'user_id': user,
         'timestamp': base_time + pd.Timedelta(days=np.random.randint(0, 60)),
         'event_type': np.random.choice(event_types, p=[0.4, 0.3, 0.2, 0.1])
    }
    )

data = pd.DataFrame(simulated)
data.head()


,user_id,timestamp,event_type
0,U6516,2024-02-21,Logout
1,U6516,2024-02-12,View_Product
2,U6516,2024-01-21,Login
3,U6516,2024-01-19,Login
4,U6516,2024-01-11,View_Product


In [ ]:
data.groupby('user_id').size().describe()
data['event_type'].value_counts()

,count
event_type,
Login,195
Purchase,132
View_Product,97
Logout,61


In [ ]:
import numpy as np

np.random.seed(42)
user_ids = data['user_id'].unique()

#simulating randomized assignment to approximate an A/B test (though in real life exposure would be non-random)
treatment_users = np.random.choice(user_ids, size=int(0.5*len(user_ids)), replace=False)

data['treatment'] = data['user_id'].isin(treatment_users).astype(int)
data.head()

,user_id,timestamp,event_type,treatment
0,U6516,2024-02-21,Logout,1
1,U6516,2024-02-12,View_Product,1
2,U6516,2024-01-21,Login,1
3,U6516,2024-01-19,Login,1
4,U6516,2024-01-11,View_Product,1


**3. Cohort Construction**
- pre-feature users = control
- post-feature users = treatment

In [ ]:
data['treatment'].value_counts()


,user_id,timestamp,event_type,treatment
0,U6516,2024-02-21,Logout,1
1,U6516,2024-02-12,View_Product,1
2,U6516,2024-01-21,Login,1
3,U6516,2024-01-19,Login,1
4,U6516,2024-01-11,View_Product,1


**4. Analysis**

Compute:
- retention rate (control vs treatment)
- difference
- confidence interval
- p-value

In [ ]:
# Ensure 'timestamp' is in datetime format
data['timestamp'] = pd.to_datetime(data['timestamp'])

# Calculate the first activity timestamp for each user
first_activity = data.groupby('user_id')['timestamp'].min().reset_index()
first_activity.rename(columns={'timestamp': 'first_activity_timestamp'}, inplace=True)

# Merge the first activity timestamp back into the main DataFrame
data = pd.merge(data, first_activity, on='user_id', how='left')

# Calculate 'days_since' the first activity for each event
data['days_since'] = (data['timestamp'] - data['first_activity_timestamp']).dt.days

data.head()


,user_id,timestamp,event_type,treatment,first_activity_timestamp,days_since
0,U6516,2024-02-21,Logout,1,2024-01-03,49
1,U6516,2024-02-12,View_Product,1,2024-01-03,40
2,U6516,2024-01-21,Login,1,2024-01-03,18
3,U6516,2024-01-19,Login,1,2024-01-03,16
4,U6516,2024-01-11,View_Product,1,2024-01-03,8


In [ ]:
retained_users = data[
    (data['days_since'] > 0) &
    (data['days_since'] <= 30)
].groupby('user_id').size()


total_users = data['user_id'].nunique() # Define total_users to include all unique users
retention_rate = len(retained_users) / total_users
print(retention_rate)


0.8181818181818182


In [ ]:
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

# Calculate retention for the treatment group
treatment_data = data[data['treatment'] == 1]
retained_treatment_users = treatment_data[
    (treatment_data['days_since'] > 0) &
    (treatment_data['days_since'] <= 30)
]['user_id'].unique()
total_treatment_users = treatment_data['user_id'].nunique()
retention_treatment = len(retained_treatment_users) / total_treatment_users if total_treatment_users > 0 else 0

# Calculate retention for the control group
control_data = data[data['treatment'] == 0]
retained_control_users = control_data[
    (control_data['days_since'] > 0) &
    (control_data['days_since'] <= 30)
]['user_id'].unique()
total_control_users = control_data['user_id'].nunique()
retention_control = len(retained_control_users) / total_control_users if total_control_users > 0 else 0

print(f"Retention Rate (Treatment Group): {retention_treatment:.4f}")
print(f"Retention Rate (Control Group): {retention_control:.4f}")

# Perform Z-test for proportions
count = np.array([len(retained_treatment_users), len(retained_control_users)])
nobs = np.array([total_treatment_users, total_control_users])

stat, pval = proportions_ztest(count, nobs)

print(f"\nZ-statistic: {stat:.4f}")
print(f"P-value: {pval:.4f}")

# Calculate confidence interval for the difference in proportions
# Difference in proportions
diff = retention_treatment - retention_control

# Standard error of the difference
se = np.sqrt(
    (retention_treatment * (1 - retention_treatment) / total_treatment_users) +
    (retention_control * (1 - retention_control) / total_control_users)
)

# Z-score for 95% confidence (1.96)
z_score = 1.96

# Confidence interval
lower_bound = diff - z_score * se
upper_bound = diff + z_score * se

print(f"\nDifference in Retention Rates: {diff:.4f}")
print(f"95% Confidence Interval for Difference: ({lower_bound:.4f}, {upper_bound:.4f})")


Retention Rate (Treatment Group): 0.8367
Retention Rate (Control Group): 0.8000

Z-statistic: 0.4738
P-value: 0.6356

Difference in Retention Rates: 0.0367
95% Confidence Interval for Difference: (-0.1149, 0.1884)


**5. Segmentation**
Break into:

new vs returning users
high vs low activity

Then compute retention again.

In [ ]:
print("\n--- 5. Segmentation Analysis ---")

# 1. Calculate user-level activity summaries
user_activity_summary = data.groupby('user_id').agg(
    max_days_since=('days_since', 'max'),
    total_events=('event_type', 'size'),
    treatment=('treatment', 'first') # Assuming treatment is constant per user
).reset_index()

# 2. Define segmentation criteria
# For 'new vs returning', use total_events.
# For 'high vs low activity', use total_events as a proxy for engagement volume.

# Let's inspect distributions to set thresholds
print("\nUser Activity Span (max_days_since) Distribution:")
print(user_activity_summary['max_days_since'].describe())

print("\nUser Total Events Distribution:")
print(user_activity_summary['total_events'].describe())

# Thresholds (can be adjusted based on data distribution or business knowledge)
# MAX_DAYS_SINCE_THRESHOLD is now descriptive of activity span, not used for New/Returning classification
MAX_DAYS_SINCE_THRESHOLD = user_activity_summary['max_days_since'].median()
TOTAL_EVENTS_THRESHOLD = user_activity_summary['total_events'].median() # Using median as a cutoff

print(f"\nUsing max_days_since median as a reference for activity span: {MAX_DAYS_SINCE_THRESHOLD} days")
print(f"Using total_events median as Low/High Activity threshold: {TOTAL_EVENTS_THRESHOLD} events")

# Assign segments to each user based on new criteria
# 'New' users have exactly 1 event, 'Returning' users have more than 1 event
user_activity_summary['user_type_segment'] = user_activity_summary['total_events'].apply(
    lambda x: 'New' if x == 1 else 'Returning'
)
user_activity_summary['activity_level_segment'] = user_activity_summary['total_events'].apply(
    lambda x: 'Low Activity' if x <= TOTAL_EVENTS_THRESHOLD else 'High Activity'
)

# Function to calculate retention metrics for a given segment
def calculate_segment_retention(segment_df, segment_name):
    print(f"\n--- {segment_name} Retention Analysis ---")

    # Calculate retention for the treatment group within the segment
    treatment_segment_data = segment_df[segment_df['treatment'] == 1]
    retained_treatment_users = treatment_segment_data[
        (treatment_segment_data['days_since'] > 0) &
        (treatment_segment_data['days_since'] <= 30)
    ]['user_id'].unique()
    total_treatment_users = treatment_segment_data['user_id'].nunique()
    retention_treatment = len(retained_treatment_users) / total_treatment_users if total_treatment_users > 0 else 0

    # Calculate retention for the control group within the segment
    control_segment_data = segment_df[segment_df['treatment'] == 0]
    retained_control_users = control_segment_data[
        (control_segment_data['days_since'] > 0) &
        (control_segment_data['days_since'] <= 30)
    ]['user_id'].unique()
    total_control_users = control_segment_data['user_id'].nunique()
    retention_control = len(retained_control_users) / total_control_users if total_control_users > 0 else 0

    print(f"Retention Rate (Treatment Group): {retention_treatment:.4f}")
    print(f"Retention Rate (Control Group): {retention_control:.4f}")

    # Perform Z-test for proportions
    if total_treatment_users > 0 and total_control_users > 0 and (len(retained_treatment_users) < total_treatment_users or len(retained_control_users) < total_control_users):
        count = np.array([len(retained_treatment_users), len(retained_control_users)])
        nobs = np.array([total_treatment_users, total_control_users])
        stat, pval = proportions_ztest(count, nobs)
        print(f"Z-statistic: {stat:.4f}")
        print(f"P-value: {pval:.4f}")
    else:
        print("Not enough data in one or both groups for Z-test or all users retained/not retained.")
        pval = np.nan

    # Calculate confidence interval for the difference in proportions
    diff = retention_treatment - retention_control
    if total_treatment_users > 0 and total_control_users > 0 and retention_treatment > 0 and retention_treatment < 1 and retention_control > 0 and retention_control < 1:
        se = np.sqrt(
            (retention_treatment * (1 - retention_treatment) / total_treatment_users) +
            (retention_control * (1 - retention_control) / total_control_users)
        )
        z_score = 1.96 # For 95% confidence
        lower_bound = diff - z_score * se
        upper_bound = diff + z_score * se
        print(f"Difference in Retention Rates: {diff:.4f}")
        print(f"95% Confidence Interval for Difference: ({lower_bound:.4f}, {upper_bound:.4f})")
    else:
        print("Cannot calculate CI for difference due to zero or one retention rate (division by zero or perfect retention/non-retention).")

    return {
        'segment': segment_name,
        'retention_treatment': retention_treatment,
        'retention_control': retention_control,
        'diff': diff,
        'pval': pval
    }

segment_results = []

# --- New vs Returning Segmentation (based on total_events) ---
new_user_ids = user_activity_summary[user_activity_summary['user_type_segment'] == 'New']['user_id'].unique()
new_users_data = data[data['user_id'].isin(new_user_ids)]
segment_results.append(calculate_segment_retention(new_users_data, "New Users"))

returning_user_ids = user_activity_summary[user_activity_summary['user_type_segment'] == 'Returning']['user_id'].unique()
returning_users_data = data[data['user_id'].isin(returning_user_ids)]
segment_results.append(calculate_segment_retention(returning_users_data, "Returning Users"))

# --- High vs Low Activity Segmentation (based on total_events median split) ---
low_activity_user_ids = user_activity_summary[user_activity_summary['activity_level_segment'] == 'Low Activity']['user_id'].unique()
low_activity_data = data[data['user_id'].isin(low_activity_user_ids)]
segment_results.append(calculate_segment_retention(low_activity_data, "Low Activity Users"))

high_activity_user_ids = user_activity_summary[user_activity_summary['activity_level_segment'] == 'High Activity']['user_id'].unique()
high_activity_data = data[data['user_id'].isin(high_activity_user_ids)]
segment_results.append(calculate_segment_retention(high_activity_data, "High Activity Users"))



--- 5. Segmentation Analysis ---

User Activity Span (max_days_since) Distribution:
count    99.000000
mean     33.737374
std      18.416710
min       0.000000
25%      23.500000
50%      40.000000
75%      49.000000
max      58.000000
Name: max_days_since, dtype: float64

User Total Events Distribution:
count    99.00000
mean      4.89899
std       2.63994
min       1.00000
25%       3.00000
50%       5.00000
75%       7.50000
max       9.00000
Name: total_events, dtype: float64

Using max_days_since median as a reference for activity span: 40.0 days
Using total_events median as Low/High Activity threshold: 5.0 events

--- New Users Retention Analysis ---
Retention Rate (Treatment Group): 0.0000
Retention Rate (Control Group): 0.0000
Z-statistic: nan
P-value: nan
Cannot calculate CI for difference due to zero or one retention rate (division by zero or perfect retention/non-retention).

--- Returning Users Retention Analysis ---
Retention Rate (Treatment Group): 0.9111
Retention Rate 

/usr/local/lib/python3.12/dist-packages/statsmodels/stats/weightstats.py:792: RuntimeWarning: invalid value encountered in scalar divide
  zstat = value / std


6. **Limitations of naive comparison:**
This analysis assumes randomized assignment between treatment and control groups. While I simulate this here, in real-world settings feature exposure is often non-random, which can introduce selection bias if certain types of users are more likely to receive the feature.

Additionally, the analysis assumes independence of observations. In practice, user behavior may be influenced by confounding factors such as geography, device type, or baseline engagement levels, which could bias the estimated treatment effect.

To address these limitations, I would apply causal inference techniques such as propensity score weighting (IPTW) to better balance the treatment and control groups and obtain more reliable estimates of feature impact.

7. **Decision**
I would recommend shipping the feature for new users, where we observe a statistically significant ~10 percentage point increase in 30-day retention (p < 0.05). Improvements in early retention are typically correlated with increased long-term user value, making this a meaningful positive signal.

However, I would recommend a segmented rollout rather than a full launch, as the effect on returning users is inconclusive and may negatively impact engagement.

As a next step, I would validate these findings in a controlled production experiment and further investigate the behavioral differences between new and returning users to understand why the feature is less effective for the latter group.